In [1]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
#Set Base Path
BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)

Connected: /content/drive/MyDrive/Text Miners/Actual Work


In [3]:
#Confirm connection
import os
print(os.listdir(BASE))

['data', 'pre-processing', 'features', 'task1-sentiments', 'task2-topics', 'task3-combination', 'dashboard', 'BeforeYouStart.docx']


In [4]:
#Install required libraries
!pip install numpy pandas scikit-learn ftfy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.5 MB/s eta 0:00:00


In [5]:
#Load Data
import pandas as pd

df = pd.read_csv(f'{BASE}/data/MergedCleaned.csv')
print('Shape:', df.shape)
print(df.head(3))

Shape: (380505, 4)
   listing_id neighbourhood        room_type  \
0       27934   Ratchathewi  Entire home/apt   
1       27934   Ratchathewi  Entire home/apt   
2       27934   Ratchathewi  Entire home/apt   

                                            comments  
0  We stayed in the apartment for a week and we e...  
1  My girlfriend and I recently stayed in Nuttee'...  
2  I stayed for one month at the condo and was re...  


In [6]:
#Import preprocess.py
import sys
sys.path.append(f'{BASE}/pre-processing')
import preprocess
print('preprocess.py loaded')

preprocess.py loaded


In [7]:
# This adds two new columns to your dataframe:
#   cleaned_text — preprocessed review as a string
#   tokens       — preprocessed review as a list of words

df = preprocess.preprocess_dataframe(df)
print(df[['comments', 'cleaned_text', 'tokens']].head(3))

                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   
2  I stayed for one month at the condo and was re...   

                                        cleaned_text  \
0  stayed apartment week enjoyed very much nuttee...   
1  girlfriend recently stayed nuttee condo month ...   
2  stayed one month condo realy pleased condo 19t...   

                                              tokens  
0  [stayed, apartment, week, enjoyed, very, much,...  
1  [girlfriend, recently, stayed, nuttee, condo, ...  
2  [stayed, one, month, condo, realy, pleased, co...  


In [8]:
#Export cleaned reviews
df.to_csv('reviews_clean.csv', index=False)
print(f"Saved reviews_clean.csv with {len(df)} rows")
print(df[['comments', 'cleaned_text']].head(3))

Saved reviews_clean.csv with 380505 rows
                                            comments  \
0  We stayed in the apartment for a week and we e...   
1  My girlfriend and I recently stayed in Nuttee'...   
2  I stayed for one month at the condo and was re...   

                                        cleaned_text  
0  stayed apartment week enjoyed very much nuttee...  
1  girlfriend recently stayed nuttee condo month ...  
2  stayed one month condo realy pleased condo 19t...  


In [9]:
#BoW Matrix
from sklearn.feature_extraction.text import CountVectorizer
import pickle, os

#Only keep the 10,000 most frequent words and ignore words that appear in fewer than 2 reviews
bow_vectorizer = CountVectorizer(max_features=10000, min_df=2, token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')  # only words with 2+ letters, must start with a letter
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_text'])

print(f"BoW matrix shape: {bow_matrix.shape}")
#Shape = (num_reviews, num_unique_words)

BoW matrix shape: (380505, 10000)


In [10]:
#TF-IDF Matrix
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=10000, min_df=2, token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (380505, 10000)


In [11]:
#Export all feature files to Google Drive /features/ folder
import os, pickle

FEATURES_PATH = os.path.join(BASE, 'features')

with open(os.path.join(FEATURES_PATH, 'bow_matrix.pkl'), 'wb') as f:
    pickle.dump(bow_matrix, f)

with open(os.path.join(FEATURES_PATH, 'bow_vectorizer.pkl'), 'wb') as f:
    pickle.dump(bow_vectorizer, f)

with open(os.path.join(FEATURES_PATH, 'tfidf_matrix.pkl'), 'wb') as f:
    pickle.dump(tfidf_matrix, f)

with open(os.path.join(FEATURES_PATH, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

print("Saved to:", FEATURES_PATH)
print("Files:", os.listdir(FEATURES_PATH))

Saved to: /content/drive/MyDrive/Text Miners/Actual Work/features
Files: ['Copy of features_tfidf.ipynb', 'bow_matrix.pkl', 'bow_vectorizer.pkl', 'tfidf_matrix.pkl', 'tfidf_vectorizer.pkl']


In [12]:
#Sanity Check
import numpy as np

#Check a few top words
bow_vocab = bow_vectorizer.get_feature_names_out()
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

print("Sample BoW vocab:", bow_vocab[:10])
print("Sample TF-IDF vocab:", tfidf_vocab[:10])

#Peek at TF-IDF scores for first review
first_row = tfidf_matrix[0]
top_indices = np.argsort(first_row.toarray()[0])[::-1][:5]
print("\nTop TF-IDF words in review 0:", tfidf_vocab[top_indices])

Sample BoW vocab: ['aaa' 'ab' 'aback' 'abandoned' 'abd' 'abe' 'ability' 'abit' 'able' 'abnb']
Sample TF-IDF vocab: ['aaa' 'ab' 'aback' 'abandoned' 'abd' 'abe' 'ability' 'abit' 'able' 'abnb']

Top TF-IDF words in review 0: ['nuttee' 'central' 'min' 'apartment' 'very']
